### def train_bpe()

- Input

    - input_path

    - vocab_size

    - special_tokens

- Output

    - vocab: dict[int, bytes]

    - merges: list[tuple[bytes, bytes]]

#### 前期已有的基础工作 - 参考Assignment1-Problem.ipynb

In [34]:
import regex as re
from collections import Counter

In [1]:
vocab = {i:bytes([i]) for i in range(256)}
vocab

{0: b'\x00',
 1: b'\x01',
 2: b'\x02',
 3: b'\x03',
 4: b'\x04',
 5: b'\x05',
 6: b'\x06',
 7: b'\x07',
 8: b'\x08',
 9: b'\t',
 10: b'\n',
 11: b'\x0b',
 12: b'\x0c',
 13: b'\r',
 14: b'\x0e',
 15: b'\x0f',
 16: b'\x10',
 17: b'\x11',
 18: b'\x12',
 19: b'\x13',
 20: b'\x14',
 21: b'\x15',
 22: b'\x16',
 23: b'\x17',
 24: b'\x18',
 25: b'\x19',
 26: b'\x1a',
 27: b'\x1b',
 28: b'\x1c',
 29: b'\x1d',
 30: b'\x1e',
 31: b'\x1f',
 32: b' ',
 33: b'!',
 34: b'"',
 35: b'#',
 36: b'$',
 37: b'%',
 38: b'&',
 39: b"'",
 40: b'(',
 41: b')',
 42: b'*',
 43: b'+',
 44: b',',
 45: b'-',
 46: b'.',
 47: b'/',
 48: b'0',
 49: b'1',
 50: b'2',
 51: b'3',
 52: b'4',
 53: b'5',
 54: b'6',
 55: b'7',
 56: b'8',
 57: b'9',
 58: b':',
 59: b';',
 60: b'<',
 61: b'=',
 62: b'>',
 63: b'?',
 64: b'@',
 65: b'A',
 66: b'B',
 67: b'C',
 68: b'D',
 69: b'E',
 70: b'F',
 71: b'G',
 72: b'H',
 73: b'I',
 74: b'J',
 75: b'K',
 76: b'L',
 77: b'M',
 78: b'N',
 79: b'O',
 80: b'P',
 81: b'Q',
 82: b'R',
 83: b'

In [ ]:
special_tokens = ["<|endoftext|>","<|startoftext|>"]

In [31]:
es = re.escape(special_tokens)

In [32]:
es

'<|endoftext|>'

In [25]:
with open("/home/ygm/llm-project/CS336-From-Scratch/Assignment/assignment1-basics/tests/fixtures/tinystories_sample.txt","r",encoding="utf-8") as f:
    text = f.read()

In [26]:
text

'\nOnce upon a time there was a little boy named Ben. Ben loved to explore the world around him. He saw many amazing things, like beautiful vases that were on display in a store. One day, Ben was walking through the store when he came across a very special vase. When Ben saw it he was amazed!\nHe said, “Wow, that is a really amazing vase! Can I buy it?”\nThe shopkeeper smiled and said, “Of course you can. You can take it home and show all your friends how amazing it is!”\nSo Ben took the vase home and he was so proud of it! He called his friends over and showed them the amazing vase. All his friends thought the vase was beautiful and couldn\'t believe how lucky Ben was.\nAnd that\'s how Ben found an amazing vase in the store!\n<|endoftext|>\nOnce upon a time, there was a reliable otter named Ollie. He lived in a river with his family. They all loved to play and swim together.\nOne day, Ollie\'s mom said, "Ollie, hurry and get some fish for dinner!" Ollie swam fast to catch fish. He saw

escape只接受单个字符串

In [27]:
escaped = re.escape("<|endoftext|>")

In [28]:
escaped

'<\\|endoftext\\|>'

In [29]:
text = re.split(escaped,text)

In [30]:
text

["\nOnce upon a time there was a little boy named Ben. Ben loved to explore the world around him. He saw many amazing things, like beautiful vases that were on display in a store. One day, Ben was walking through the store when he came across a very special vase. When Ben saw it he was amazed!\nHe said, “Wow, that is a really amazing vase! Can I buy it?”\nThe shopkeeper smiled and said, “Of course you can. You can take it home and show all your friends how amazing it is!”\nSo Ben took the vase home and he was so proud of it! He called his friends over and showed them the amazing vase. All his friends thought the vase was beautiful and couldn't believe how lucky Ben was.\nAnd that's how Ben found an amazing vase in the store!\n",
 '\nOnce upon a time, there was a reliable otter named Ollie. He lived in a river with his family. They all loved to play and swim together.\nOne day, Ollie\'s mom said, "Ollie, hurry and get some fish for dinner!" Ollie swam fast to catch fish. He saw his frie

In [33]:
GPT2_PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

In [35]:
pretoken_counts = Counter()

for segment in text:
    for match in re.finditer(GPT2_PAT, segment):
        pretoken = match.group(0)
        pretoken_counts[pretoken] += 1

In [36]:
pretoken_counts

Counter({'.': 64,
         ',': 39,
         ' the': 28,
         '\n': 25,
         ' a': 24,
         ' and': 22,
         ' was': 21,
         ' to': 21,
         ' with': 12,
         ' his': 11,
         ' Lucy': 11,
         ' it': 10,
         ' day': 9,
         ' He': 8,
         ' "': 8,
         ' Bob': 8,
         ' spirit': 8,
         ' her': 8,
         ' Ben': 7,
         ' that': 7,
         ' said': 7,
         ' friends': 7,
         ' Ollie': 7,
         ' They': 7,
         ' play': 7,
         ' big': 7,
         ' not': 7,
         ' he': 6,
         ' very': 6,
         ' vase': 6,
         '!': 6,
         ' home': 6,
         ' fish': 6,
         ' The': 6,
         ' orange': 6,
         ' amazing': 5,
         ' were': 5,
         ' in': 5,
         ' family': 5,
         ' mom': 5,
         ' for': 5,
         ' dinner': 5,
         ' friend': 5,
         ' Tim': 5,
         ' tiger': 5,
         ' thumb': 5,
         'Once': 4,
         ' upon': 4,
       

In [48]:
type(text)

list

In [50]:
word_counts = Counter()

for segment in text:
    for match in re.finditer(GPT2_PAT, segment):
        pretoken = match.group(0)
        print(pretoken)
        byte_tuple = tuple(bytes([b]) for b in pretoken.encode("utf-8"))
        print(byte_tuple)
        word_counts[byte_tuple] += 1



(b'\n',)
Once
(b'O', b'n', b'c', b'e')
 upon
(b' ', b'u', b'p', b'o', b'n')
 a
(b' ', b'a')
 time
(b' ', b't', b'i', b'm', b'e')
 there
(b' ', b't', b'h', b'e', b'r', b'e')
 was
(b' ', b'w', b'a', b's')
 a
(b' ', b'a')
 little
(b' ', b'l', b'i', b't', b't', b'l', b'e')
 boy
(b' ', b'b', b'o', b'y')
 named
(b' ', b'n', b'a', b'm', b'e', b'd')
 Ben
(b' ', b'B', b'e', b'n')
.
(b'.',)
 Ben
(b' ', b'B', b'e', b'n')
 loved
(b' ', b'l', b'o', b'v', b'e', b'd')
 to
(b' ', b't', b'o')
 explore
(b' ', b'e', b'x', b'p', b'l', b'o', b'r', b'e')
 the
(b' ', b't', b'h', b'e')
 world
(b' ', b'w', b'o', b'r', b'l', b'd')
 around
(b' ', b'a', b'r', b'o', b'u', b'n', b'd')
 him
(b' ', b'h', b'i', b'm')
.
(b'.',)
 He
(b' ', b'H', b'e')
 saw
(b' ', b's', b'a', b'w')
 many
(b' ', b'm', b'a', b'n', b'y')
 amazing
(b' ', b'a', b'm', b'a', b'z', b'i', b'n', b'g')
 things
(b' ', b't', b'h', b'i', b'n', b'g', b's')
,
(b',',)
 like
(b' ', b'l', b'i', b'k', b'e')
 beautiful
(b' ', b'b', b'e', b'a', b'u', b't', 

In [38]:
word_counts

Counter({(b'.',): 64,
         (b',',): 39,
         (b' ', b't', b'h', b'e'): 28,
         (b'\n',): 25,
         (b' ', b'a'): 24,
         (b' ', b'a', b'n', b'd'): 22,
         (b' ', b'w', b'a', b's'): 21,
         (b' ', b't', b'o'): 21,
         (b' ', b'w', b'i', b't', b'h'): 12,
         (b' ', b'h', b'i', b's'): 11,
         (b' ', b'L', b'u', b'c', b'y'): 11,
         (b' ', b'i', b't'): 10,
         (b' ', b'd', b'a', b'y'): 9,
         (b' ', b'H', b'e'): 8,
         (b' ', b'"'): 8,
         (b' ', b'B', b'o', b'b'): 8,
         (b' ', b's', b'p', b'i', b'r', b'i', b't'): 8,
         (b' ', b'h', b'e', b'r'): 8,
         (b' ', b'B', b'e', b'n'): 7,
         (b' ', b't', b'h', b'a', b't'): 7,
         (b' ', b's', b'a', b'i', b'd'): 7,
         (b' ', b'f', b'r', b'i', b'e', b'n', b'd', b's'): 7,
         (b' ', b'O', b'l', b'l', b'i', b'e'): 7,
         (b' ', b'T', b'h', b'e', b'y'): 7,
         (b' ', b'p', b'l', b'a', b'y'): 7,
         (b' ', b'b', b'i', b'g'): 7,
  

In [39]:
pair_counts = Counter()
for word, freq in word_counts.items():
    if len(word) < 2:
        continue
    for pair in zip(word,word[1:]):
        pair_counts[pair] += freq

In [40]:
pair_counts

Counter({(b' ', b't'): 97,
         (b'h', b'e'): 78,
         (b't', b'h'): 73,
         (b' ', b'a'): 72,
         (b' ', b's'): 61,
         (b'e', b'r'): 52,
         (b' ', b'h'): 52,
         (b' ', b'w'): 47,
         (b'n', b'd'): 45,
         (b'e', b'd'): 42,
         (b' ', b'f'): 42,
         (b'a', b'n'): 40,
         (b'e', b'n'): 38,
         (b't', b'o'): 38,
         (b'i', b't'): 36,
         (b'a', b's'): 32,
         (b'i', b'n'): 31,
         (b'r', b'e'): 29,
         (b' ', b'd'): 29,
         (b'r', b'i'): 29,
         (b'i', b's'): 28,
         (b' ', b'b'): 27,
         (b'h', b'i'): 27,
         (b'n', b'e'): 27,
         (b'i', b'e'): 27,
         (b'v', b'e'): 25,
         (b'w', b'a'): 24,
         (b' ', b'i'): 23,
         (b'l', b'l'): 23,
         (b'm', b'e'): 22,
         (b'o', b'u'): 22,
         (b'a', b'y'): 22,
         (b' ', b'l'): 21,
         (b'l', b'i'): 21,
         (b' ', b'n'): 21,
         (b'o', b'r'): 21,
         (b'h', b'a'): 21,
 

In [41]:
best_pair = max(pair_counts.items(),key=lambda x: (x[1], x[0]))[0]

In [42]:
best_pair

(b' ', b't')

In [44]:
merge_token = best_pair[0] + best_pair[1]
print(merge_token)

b' t'


In [45]:
def apply_merge_token(word_bytes_counts,best_pair: tuple):
    merge_token = best_pair[0] + best_pair[1]
    new_word_bytes_counts = Counter()
    
    for word,freq in word_bytes_counts.items():
        new_word = []
        i = 0
        
        while i < len(word):
            if i < len(word) - 1 and (word[i],word[i + 1]) == best_pair:
                new_word.append(merge_token)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_word_bytes_counts[tuple(new_word)] += freq
    return new_word_bytes_counts

In [46]:
word_counts = apply_merge_token(word_counts,best_pair)

In [47]:
word_counts

Counter({(b'.',): 64,
         (b',',): 39,
         (b' t', b'h', b'e'): 28,
         (b'\n',): 25,
         (b' ', b'a'): 24,
         (b' ', b'a', b'n', b'd'): 22,
         (b' ', b'w', b'a', b's'): 21,
         (b' t', b'o'): 21,
         (b' ', b'w', b'i', b't', b'h'): 12,
         (b' ', b'h', b'i', b's'): 11,
         (b' ', b'L', b'u', b'c', b'y'): 11,
         (b' ', b'i', b't'): 10,
         (b' ', b'd', b'a', b'y'): 9,
         (b' ', b'H', b'e'): 8,
         (b' ', b'"'): 8,
         (b' ', b'B', b'o', b'b'): 8,
         (b' ', b's', b'p', b'i', b'r', b'i', b't'): 8,
         (b' ', b'h', b'e', b'r'): 8,
         (b' ', b'B', b'e', b'n'): 7,
         (b' t', b'h', b'a', b't'): 7,
         (b' ', b's', b'a', b'i', b'd'): 7,
         (b' ', b'f', b'r', b'i', b'e', b'n', b'd', b's'): 7,
         (b' ', b'O', b'l', b'l', b'i', b'e'): 7,
         (b' ', b'T', b'h', b'e', b'y'): 7,
         (b' ', b'p', b'l', b'a', b'y'): 7,
         (b' ', b'b', b'i', b'g'): 7,
         (b' ', b

text_convert_byte

In [51]:
def byte_word_count(segments: list):
    word_counts = Counter()
    for segment in segments:
        for match in re.finditer(GPT2_PAT, segment):
            pretoken = match.group(0)
            byte_tuple = tuple(bytes([b]) for b in pretoken.encode("utf-8"))
            word_counts[byte_tuple] += 1
    return word_counts

In [52]:
counts = byte_word_count(text)

In [64]:
def split_with_special_tokens(text, special_tokens):
    if not special_tokens:
        return [text]
    
    escaped_tokens = [re.escape(special_tok) for special_tok in special_tokens]
    pattern = "|".join(sorted(escaped_tokens, key=len, reverse=True))
    segments = re.split(pattern,text)
    return [seg for seg in segments if seg]

In [57]:
smaple_str1 = (
    "low low low low low<|endoftext|>"
    "lower lower  widest widest widest<|startoftext|>"
    "newest newest newest newest newest"
)

In [58]:
special_tokens = ["<|endoftext|>","<|startoftext|>"]

In [62]:
segments = split_with_special_tokens(smaple_str1,special_tokens)

<\|startoftext\|>|<\|endoftext\|>


In [63]:
segments

['low low low low low',
 'lower lower  widest widest widest',
 'newest newest newest newest newest']

### 构造train_bpr函数

In [ ]:
def train_bpe(input_path: str, vocab_size: int, special_tokens: list[str]):
    
    # Read the text
    with open(input_path,"r",encoding="utf-8") as f:
        text = f.read()
    
    escaped_tokens = [re.escape(special_tok) for special_tok in special_tokens]
    pattern = "|".join(sorted(escaped_tokens, key=len, reverse=True))
    segments = re.split(pattern,text)
    
    word_counts = Counter()
    for segment in segments:
        for match in re.finditer(GPT2_PAT, segment):
            pretoken = match.group(0)
            byte_tuple = tuple(bytes([b]) for b in pretoken.encode("utf-8"))
            word_counts[byte_tuple] += 1
            
    
    pair_counts = Counter()
    for word, freq in word_counts.items():
        if len(word) < 2:
            continue
        for pair in zip(word,word[1:]):
            pair_counts[pair] += freq
    
    
    best_pair = max(pair_counts.items(),key=lambda x: (x[1], x[0]))[0]
    merge_token = best_pair[0] + best_pair[1]
    new_word_counts = Counter()
    
    for word,freq in word_counts.items():
        new_word = []
        i = 0
        
        while i < len(word):
            if i < len(word) - 1 and (word[i],word[i + 1]) == best_pair:
                new_word.append(merge_token)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_word_counts[tuple(new_word)] += freq
    
    vocab[vocab_size] = merge_token
    vocab_size += 1
    
    return vocab,new_word_counts